In [0]:
dbutils.widgets.text("catalog_name", "dbw_chinook_team",               "Catalog")
dbutils.widgets.text("schema_name",  "chinook_bronze",                  "Schema")
dbutils.widgets.text("base_path",    "/Volumes/dbw_chinook_team/raw_zone/chinook", "Base Path")
dbutils.widgets.text("table_name",   "all",                             "Table Name")

In [0]:
from datetime import datetime
from pyspark.sql.functions import lit, current_timestamp

CATALOG     = dbutils.widgets.get("catalog_name")
SCHEMA      = dbutils.widgets.get("schema_name")
BASE_PATH   = dbutils.widgets.get("base_path")
TABLE_PARAM = dbutils.widgets.get("table_name")
BRONZE      = f"{CATALOG}.{SCHEMA}"

exec_time = datetime.now()
YEAR  = exec_time.strftime("%Y")
MONTH = exec_time.strftime("%m")
DAY   = exec_time.strftime("%d")

print(f"✅ Parameters loaded")
print(f"   Catalog   : {CATALOG}")
print(f"   Bronze    : {BRONZE}")
print(f"   Base path : {BASE_PATH}")
print(f"   Date      : {YEAR}/{MONTH}/{DAY}")

In [0]:
# Read active tables from metadata
metadata_df = spark.table(f"{BRONZE}.pipeline_metadata") \
    .filter("active_flag = 'Y'")

if TABLE_PARAM != "all":
    metadata_df = metadata_df.filter(f"table_name = '{TABLE_PARAM}'")

tables = [(row.table_name, row.file_name) 
          for row in metadata_df.collect()]

print(f"✅ Tables to process: {len(tables)}")
for t, f in tables:
    print(f"   {t} → {f}")

In [0]:
print("\n=== EXTRACTING TO RAW ZONE ===")

for table_name, file_name in tables:
    print(f"\nProcessing: {table_name}")
    try:
        # Read via Connection Manager — no credentials!
        source_df = spark.table(
            f"chinook_sql_conn_catalog.chinook.{table_name}"
        )
        source_count = source_df.count()

        # Dynamic file path
        file_path = (f"{BASE_PATH}/{table_name.lower()}"
                    f"/{YEAR}/{MONTH}/{DAY}/{file_name}")

        # Write Parquet to Volume
        source_df.write \
            .format("parquet") \
            .mode("overwrite") \
            .save(file_path)

        # Verify
        target_count = spark.read.parquet(file_path).count()
        status = "SUCCESS" if source_count == target_count \
                           else "FAILED"

        # Log to execution table
        spark.sql(f"""
            INSERT INTO {BRONZE}.pipeline_execution_log VALUES (
                '{table_name}',
                current_timestamp(),
                '{status}',
                {source_count},
                {target_count},
                '{file_path}',
                current_timestamp()
            )
        """)

        print(f"  ✅ {table_name}: {source_count} rows → {file_path}")

    except Exception as e:
        spark.sql(f"""
            INSERT INTO {BRONZE}.pipeline_execution_log VALUES (
                '{table_name}', current_timestamp(),
                'FAILED', 0, 0, 'N/A', current_timestamp()
            )
        """)
        print(f"  ❌ {table_name} FAILED: {e}")

print("\n=== EXTRACTION COMPLETE ===")

In [0]:
print("=== EXECUTION LOG ===")
spark.table(f"{BRONZE}.pipeline_execution_log") \
    .select("table_name","status",
            "source_row_count","target_row_count") \
    .show(truncate=False)